# FunctionObject2D

A 2D shape defined by a Python function: anything that has a membership test but no primitive.

`FunctionObject2D(func, lower_bound, upper_bound)` calls `func((x, y))` for each point and
needs a bounding box, which cannot be derived from a function. It has no level set, so
distances and crossings are found by sampling its membership. Prefer a primitive or a Boolean
combination of primitives where one fits.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import FunctionObject2D, Grid
from simgeometry.crossings import crossings


def flower(coords):
    x, y = coords
    return np.hypot(x, y) < 0.8 + 0.25 * np.cos(5 * np.arctan2(y, x))


shape = FunctionObject2D(flower, lower_bound=(-1.05, -1.05), upper_bound=(1.05, 1.05))
print("centre inside:", shape.contains((0.0, 0.0)), " level set:", shape.has_level_set)
assert shape.contains((1.0, 0.0)) and not shape.contains((0.0, 1.0))

## Node mask and sampled crossings

The grid nodes inside, and the first crossing of steps from the centre outward. The crossings
are sampled along each step and refined by bisection (`exact` is false).

In [ ]:
grid = Grid(2, (-1.1, -1.1), (1.1, 1.1), (45, 45))
mask = shape.node_mask(grid).ravel()
nodes = grid.node_points()

angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
start = np.zeros((2, angles.size))
end = 1.2 * np.stack((np.cos(angles), np.sin(angles)))
hits = crossings(shape, start, end)
print("all steps leave the shape:", bool(hits.hit.all()), " exact:", hits.exact)

fig, ax = plt.subplots(figsize=(5, 5), layout="constrained")
ax.scatter(*nodes[:, mask], s=6, color="tab:purple", label="nodes inside")
for i in range(angles.size):
    ax.plot(*np.stack((start[:, i], hits.point[:, i]), axis=1), color="0.6", lw=0.8)
ax.plot(*hits.point, "o", color="tab:red", ms=4, label="crossings")
ax.set(xlabel="x", ylabel="y", title="A FunctionObject2D")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.show()